# Instagram 치이카와 중심 게시물 재정제

`instagram_normalized_3nf_posts.csv`에서 치이카와 애니메이션 캐릭터보다 다른 애니메이션·브랜드 캐릭터 IP 언급 비율이 높은 게시물을 제거한다.

- 원본 파일은 수정하지 않는다.
- 치이카와 언급 수와 타 IP 언급 수를 게시물별로 계산한다.
- `타 IP 언급 수 / (치이카와 언급 수 + 타 IP 언급 수) > 0.5`인 행만 제거한다. 즉, 동률은 보수적으로 유지한다.
- `시사`, `우사기`, `카니`처럼 일반어·인명으로도 쓰이는 이름은 다른 명확한 치이카와 신호가 함께 있을 때만 보조 점수로 반영한다.
- 결과 CSV는 원본과 동일한 7개 컬럼을 유지한다.

In [ ]:
from pathlib import Path
from collections import Counter
import re
import unicodedata

import pandas as pd

CSV_DIR = Path(r"C:\Users\user\Desktop\KDT_2026\프로젝트\02_데이터_프로젝트\02_csv\02_instagram")
INPUT_PATH = CSV_DIR / "instagram_normalized_3nf_posts.csv"
OUTPUT_PATH = CSV_DIR / "instagram_normalized_3nf_posts_chiikawa_filtered.csv"

INPUT_PATH, OUTPUT_PATH

## 1. 치이카와 및 타 IP 사전

한국어·일본어·영문 표기와 데이터에서 확인된 주요 표기 변형을 함께 사용한다. 타 IP 사전은 분석 데이터에서 반복적으로 확인된 브랜드를 중심으로 구성했다.

In [ ]:
# 브랜드명 또는 비교적 고유한 캐릭터명
CHIIKAWA_CORE_TERMS = [
    "치이카와", "치이키와", "먼작귀", "먼가작고귀여운녀석",
    "ちいかわ", "chiikawa", "chikawa", "吉伊卡哇",
    "又小又可愛", "なんか小さくてかわいいやつ", "somethingsmallandcute",
    "갑옷씨", "갑옷아저씨", "포셰트갑옷씨", "포쉐트갑옷씨",
    "데카츠요", "데카라", "でかつよ",
    "랏코", "ラッコ", "rakko",
    "모몽가", "모몬가", "モモンガ", "momonga",
    "아노코", "あのこ",
    "쿠리만쥬", "くりまんじゅう", "kurimanju", "栗子饅頭",
    "파자마파티즈", "パジャマパーティーズ",
    "하치와레", "ハチワレ", "hachiware", "小八猫", "小八貓", "哈奇喵",
]

# 일반어·인명과 충돌 가능성이 있어 core 신호가 있을 때만 점수에 더한다.
CHIIKAWA_CONTEXT_TERMS = [
    "시사", "シーサー",
    "우사기", "うさぎ", "usagi", "乌萨奇", "烏薩奇", "兔兔",
    "카니", "カニ", "kanie", "kanny",
]

OTHER_IP_TERMS = {
    "농담곰/나가노베어": ["농담곰", "nongdamgom", "自分ツッコミくま", "ナガノのくま", "나가노곰", "나가노베어"],
    "산리오": [
        "산리오", "サンリオ", "sanrio", "헬로키티", "hello kitty", "hellokitty", "ハローキティ",
        "쿠로미", "kuromi", "クロミ", "마이멜로디", "마이멜로", "my melody", "mymelody", "マイメロディ",
        "시나모롤", "시나몬롤", "cinnamoroll", "シナモロール", "폼폼푸린", "pompompurin", "ポムポムプリン",
        "포차코", "pochacco", "ポチャッコ", "한교동", "ハンギョドン", "배드바츠마루", "badtz", "バッドばつ丸",
        "우사하나", "ウサハナ", "코기뮹", "こぎみゅん", "슈가버니", "슈가바니", "シュガーバニーズ",
        "케로피", "けろけろけろっぴ", "턱시도샘", "タキシードサム", "리틀트윈스타", "키키라라",
        "リトルツインスターズ", "구데타마", "ぐでたま", "gudetama", "코로코로쿠리링", "쿠리링",
        "위시미멜", "위시마멜", "ウィッシュミーメル", "메로쿠로", "아히루노펫클", "あひるのペックル",
    ],
    "디즈니/픽사": [
        "디즈니", "disney", "ディズニー", "픽사", "pixar", "ピクサー", "미키마우스", "mickeymouse",
        "ミッキーマウス", "미니마우스", "minniemouse", "ミニーマウス", "도날드덕", "donaldduck",
        "ドナルドダック", "데이지덕", "daisyduck", "더피프렌즈", "더피", "duffy", "ダッフィー",
        "쉐리메이", "shelliemay", "シェリーメイ", "스텔라루", "stellalou", "ステラルー",
        "리나벨", "linabell", "リーナベル", "젤라토니", "gelatoni", "ジェラトーニ",
        "올루멜", "olumel", "オルメル", "겨울왕국", "frozen", "アナと雪の女王",
        "토이스토리", "toystory", "トイストーリー", "몬스터주식회사", "monstersinc", "モンスターズインク",
        "라푼젤", "rapunzel", "ラプンツェル", "곰돌이푸", "winniethepooh", "くまのプーさん",
        "주토피아", "zootopia", "ズートピア", "칩앤데일", "chipndale", "チップとデール",
        "라이온킹", "lionking", "인어공주", "리틀머메이드", "リトルマーメイド",
        "릴로앤스티치", "liloandstitch", "스티치", "アリストキャット", "아리스토캣", "유니베어시티",
    ],
    "포켓몬": ["포켓몬", "pokemon", "pokémon", "ポケモン", "피카츄", "pikachu", "ピカチュウ", "이브이", "eevee", "イーブイ", "잠만보", "snorlax", "カビゴン", "메타몽", "ditto", "メタモン"],
    "닌텐도/게임": ["닌텐도월드", "슈퍼마리오", "supermario", "スーパーマリオ", "마리오브라더스", "루이지", "luigi", "별의커비", "커비", "kirby", "カービィ", "젤다의전설", "legendofzelda", "ゼルダの伝説", "스플래툰", "splatoon", "スプラトゥーン", "동물의숲", "animalcrossing", "どうぶつの森", "피크민", "pikmin", "ピクミン", "소닉더헤지혹", "sonicthehedgehog", "ソニック", "뿌요뿌요", "puyopuyo", "ぷよぷよ"],
    "지브리": ["지브리", "ghibli", "ジブリ", "토토로", "totoro", "トトロ", "가오나시", "カオナシ", "하울의움직이는성", "howlsmovingcastle", "ハウルの動く城", "센과치히로", "千と千尋", "마녀배달부키키", "魔女の宅急便", "포뇨", "ponyo", "ポニョ"],
    "스누피/피너츠": ["스누피", "snoopy", "スヌーピー", "피너츠", "peanuts", "찰리브라운", "charliebrown", "チャーリーブラウン", "우드스탁", "woodstock"],
    "산엑스": ["산엑스", "san-x", "sanx", "리락쿠마", "rilakkuma", "リラックマ", "코리락쿠마", "korilakkuma", "コリラックマ", "키이로이토리", "キイロイトリ", "스미코구라시", "sumikkogurashi", "すみっコぐらし"],
    "미피": ["미피", "miffy", "ミッフィー", "nijntje"],
    "무민": ["무민", "moomin", "ムーミン"],
    "도라에몽": ["도라에몽", "doraemon", "ドラえもん"],
    "짱구": ["짱구는못말려", "짱구", "crayonshinchan", "クレヨンしんちゃん", "신짱구"],
    "호빵맨": ["호빵맨", "앙팡맨", "anpanman", "アンパンマン"],
    "미니언즈": ["미니언즈", "minions", "ミニオンズ"],
    "오빤쮸우사기": ["오빤쮸우사기", "오판츄우사기", "오판츄", "opanchuusagi", "ophantz", "おぱんちゅうさぎ", "オパンチュ"],
    "모후산도": ["모후산도", "모프샌드", "모후샌드", "mofusand", "moffsand", "モフサンド"],
    "카카오/라인프렌즈": ["카카오프렌즈", "kakaofriends", "춘식이", "라인프렌즈", "linefriends", "브라운앤프렌즈", "bt21"],
    "잔망루피": ["잔망루피", "zanmangloopy"],
    "케어베어": ["케어베어", "carebears", "carebear", "ケアベア"],
    "몰랑": ["몰랑이", "몰랑", "molang"],
    "기타 캐릭터": [
        "티니핑", "캐치티니핑", "몬치치", "monchhichi", "モンチッチ", "시로탄", "しろたん",
        "스폰지밥", "spongebob", "위베어베어스", "webarebears", "패딩턴", "paddington",
        "페파피그", "peppapig", "세서미스트리트", "sesamestreet", "텔레토비", "teletubbies",
        "다마고치", "tamagotchi", "たまごっち", "보노보노", "ぼのぼの", "파워퍼프걸",
        "powerpuffgirls", "가필드", "garfield", "오바케누", "オバケーヌ", "오분구", "obungu",
        "お文具といっしょ", "코우펜짱", "コウペンちゃん", "코지코지", "コジコジ",
    ],
    "일본 애니메이션": [
        "귀멸의칼날", "귀칼", "demonslayer", "鬼滅の刃", "주술회전", "jujutsukaisen", "呪術廻戦",
        "원피스", "onepiece", "ワンピース", "나루토", "naruto", "ナルト", "명탐정코난",
        "detectiveconan", "名探偵コナン", "스파이패밀리", "spyxfamily", "체인소맨", "チェンソーマン",
        "슬램덩크", "slamdunk", "세일러문", "セーラームーン", "드래곤볼", "dragonball",
        "ドラゴンボール", "하이큐", "haikyu", "ハイキュー", "최애의아이", "推しの子",
        "히로아카", "僕のヒーローアカデミア", "진격의거인", "進撃の巨人", "꼬마마법사레미",
    ],
}

len(CHIIKAWA_CORE_TERMS), sum(map(len, OTHER_IP_TERMS.values())), len(OTHER_IP_TERMS)

In [ ]:
def normalize_text(value):
    if pd.isna(value):
        return ""
    return unicodedata.normalize("NFKC", str(value)).lower().replace("_", " ")


def compile_terms(terms):
    normalized = [normalize_text(term) for term in terms]
    # 긴 표현을 먼저 매칭해 '헬로키티'와 '키티'처럼 겹치는 표현의 중복 집계를 방지한다.
    alternatives = sorted(set(normalized), key=len, reverse=True)
    return re.compile("|".join(map(re.escape, alternatives)), flags=re.IGNORECASE)


CORE_PATTERN = compile_terms(CHIIKAWA_CORE_TERMS)
CONTEXT_PATTERN = compile_terms(CHIIKAWA_CONTEXT_TERMS)
OTHER_PATTERNS = {group: compile_terms(terms) for group, terms in OTHER_IP_TERMS.items()}


def count_matches(text, pattern):
    return len(pattern.findall(text))


def score_post(description):
    text = normalize_text(description)
    core_count = count_matches(text, CORE_PATTERN)
    context_count = count_matches(text, CONTEXT_PATTERN) if core_count > 0 else 0
    other_counts = {group: count_matches(text, pattern) for group, pattern in OTHER_PATTERNS.items()}
    chiikawa_count = core_count + context_count
    other_count = sum(other_counts.values())
    total_ip_count = chiikawa_count + other_count
    other_ratio = other_count / total_ip_count if total_ip_count else 0.0
    dominant_other_ip = max(other_counts, key=other_counts.get) if other_count else None
    return chiikawa_count, other_count, other_ratio, dominant_other_ip


## 2. 게시물별 IP 언급 비율 계산

타 IP 비율이 50%를 초과할 때만 제거한다. 치이카와와 타 IP가 같은 횟수로 언급된 콜라보·비교 게시물은 유지한다.

In [ ]:
df = pd.read_csv(INPUT_PATH)
original_columns = df.columns.tolist()

scores = df["description"].apply(score_post).apply(pd.Series)
scores.columns = ["chiikawa_mentions", "other_ip_mentions", "other_ip_ratio", "dominant_other_ip"]

review_df = pd.concat([df, scores], axis=1)
review_df["remove_other_ip_dominant"] = review_df["other_ip_ratio"] > 0.5

filtered_df = review_df.loc[~review_df["remove_other_ip_dominant"], original_columns].copy()
removed_df = review_df.loc[review_df["remove_other_ip_dominant"]].copy()

summary = pd.Series({
    "원본 행 수": len(df),
    "제거 행 수": len(removed_df),
    "유지 행 수": len(filtered_df),
    "제거율(%)": round(len(removed_df) / len(df) * 100, 2),
    "동률로 유지한 행 수": int(((review_df["other_ip_mentions"] > 0) & (review_df["other_ip_mentions"] == review_df["chiikawa_mentions"])).sum()),
})
summary

In [ ]:
# 제거된 게시물의 대표 타 IP 분포
removed_by_ip = (
    removed_df["dominant_other_ip"]
    .value_counts(dropna=False)
    .rename_axis("대표 타 IP")
    .to_frame("제거 행 수")
)
removed_by_ip.head(15)

In [ ]:
# 경계 사례 검토: 타 IP 비율이 50%에 가까운 제거 행
borderline_removed = removed_df.loc[
    removed_df["other_ip_ratio"].between(0.5, 0.67, inclusive="right"),
    ["post_id", "chiikawa_mentions", "other_ip_mentions", "other_ip_ratio", "dominant_other_ip", "description"],
].sort_values(["other_ip_ratio", "post_id"])
borderline_removed.head(10)

## 3. 결과 저장 및 검증

분석용 후속 코드와의 호환성을 위해 진단 컬럼은 CSV에 넣지 않고, 원본 컬럼과 순서만 저장한다.

In [ ]:
filtered_df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")

# 저장 결과를 다시 읽어 스키마·행 수·고유키·원본 보존을 확인한다.
saved_df = pd.read_csv(OUTPUT_PATH)
assert INPUT_PATH.exists(), "원본 파일이 존재하지 않습니다."
assert OUTPUT_PATH.exists(), "결과 파일이 생성되지 않았습니다."
assert INPUT_PATH.resolve() != OUTPUT_PATH.resolve(), "원본 경로에 덮어쓸 수 없습니다."
assert saved_df.columns.tolist() == original_columns, "결과 CSV의 컬럼 또는 순서가 원본과 다릅니다."
assert len(saved_df) == len(filtered_df), "저장 전후 행 수가 다릅니다."
pd.testing.assert_frame_equal(
    saved_df.reset_index(drop=True),
    filtered_df.reset_index(drop=True),
    check_dtype=False,
)
assert saved_df["post_id"].is_unique == df["post_id"].is_unique, "post_id 고유성 상태가 달라졌습니다."
assert set(saved_df["post_id"]).issubset(set(df["post_id"])), "원본에 없던 post_id가 결과에 포함됐습니다."

{
    "output_path": str(OUTPUT_PATH),
    "rows": len(saved_df),
    "columns": saved_df.columns.tolist(),
    "file_size_bytes": OUTPUT_PATH.stat().st_size,
}